# (*)ARCH Model (package [`sktime`](https://www.sktime.net/en/stable/index.html))  

In [ ]:
import numpy as np
import pandas as pd

from sktime.forecasting.arch import ARCH
from sktime.utils.plotting import plot_series
# forecasting horizon
from sktime.forecasting.base import ForecastingHorizon

import pandas_datareader.data as web

# visualization settings
import matplotlib.pyplot as plt

# Do not show warnings
import warnings
warnings.simplefilter(action='ignore', category=Warning)

## Fitting (*)ARCH of a given order

Use class [`ARCH`](https://www.sktime.net/en/stable/api_reference/auto_generated/sktime.forecasting.arch.ARCH.html)

Basic parameters 

|Parameter|Description|Values|
|-|-|-|
|`mean`|Mean model|'Constant’, ‘Zero’, ‘LS’, ‘AR’, ‘ARX’, ‘HAR’ and ‘HARX’|
|`lags`|Lags for the mean model|int|
|`vol`|Volatility model|'GARCH’ (default), ‘ARCH’, ‘EGARCH’, ‘FIARCH’ and ‘HARCH’|
|`p`, `o`, `q`|Lags for the volatility model|int|
|`power`|Power for volatility|int|
|`dist`|Distribution for normalized shocks|‘normal’/‘gaussian’ (default), ‘t’/‘studentst’, 'skewstudent’/‘skewt’, 'ged’/‘generalized error'|

Load weekly data from the [`FRED`](https://fred.stlouisfed.org/) database on Market Yield on U.S. Treasury Securities at 10-Year Constant Maturity (Symbol [`WGS10YR`](https://fred.stlouisfed.org/series/WGS10YR)) from 2000-01-01 to 2023-12-31 and create dataframe `y` - first difference

In [ ]:
rate = web.DataReader(name='WGS10YR', data_source='fred', start='2000-01-01', end='2023-12-31')
y = rate.diff().dropna()

plot_series(y)
plt.show()

### Symmetric GARCH
Fit AR(1)-GARCH(1,0,1) model. By default, we assume that normalized shocks have a standard Gaussian distribution

Specification

\begin{align*}
	y_t&=\alpha_0+\phi_1y_{t-1}+u_t \\ u_t&=\sigma_t \epsilon_t \\
	\sigma^\lambda_t &=\omega+\alpha|u_{t-1}|^\lambda+\beta\sigma^\lambda_{t-1} \\
	\epsilon_t&\sim \mathcal{N}(0,1)
\end{align*}

where $\lambda=2$

In [ ]:
forecaster = ARCH(mean='AR', lags=1, vol='GARCH', p=1, q=1, power=2)  
forecaster.fit(y)

forecaster.summary()

In [ ]:
forecaster.get_fitted_params()

### Asymmetric GARCH

Fit AR(2)-GARCH(1,1,1) model. By default, we assume that normalized shocks have a _t_ distribution

Specification

\begin{align*}
	y_t&=\alpha_0+\phi_1y_{t-1}+\phi_2y_{t-2}+u_t \\ u_t&=\sigma_t \epsilon_t \\
	\sigma^\lambda_t &=\omega+\alpha|u_{t-1}|^\lambda+\gamma|u_{t-1}|^\lambda I(u_{t-1}<0)+\beta\sigma^\lambda_{t-1} \\
	\epsilon_t&\sim \mathcal{N}(0,1)
\end{align*}

where $\lambda=1$



In [ ]:
forecaster = ARCH(mean='AR', lags=2, vol='GARCH', p=1, o=1, q=1, power=1)  
forecaster.fit(y)

forecaster.summary()

### Symmetric EGARCH

Fit AR(1)-EGARCH(1,0,1) model. By default, we assume that normalized shocks have a standard Gaussian distribution

Specification

\begin{align*}
	y_t&=\alpha_0+\phi_1y_{t-1}+u_t \\ u_t&=\sigma_t \epsilon_t \\
	\log \sigma^2_t &=\omega+\alpha\left(|u_{t-1}|-\sqrt{2/\pi}\right)+\beta\log\sigma^2_{t-1} \\
	\epsilon_t&\sim \mathcal{N}(0,1)
\end{align*}

In [ ]:
forecaster = ARCH(mean='AR', lags=1, vol='EGARCH', p=1, q=1)  
forecaster.fit(y)

forecaster.summary()

### Asymmetric EGARCH

Fit AR(2)-EGARCH(1,1,1) model. By default, we assume that normalized shocks have a standard Gaussian distribution

Specification

\begin{align*}
	y_t&=\alpha_0+\phi_1y_{t-1}+\phi_2y_{t-2}+u_t \\ u_t&=\sigma_t \epsilon_t \\
	\log \sigma^2_t &=\omega+\alpha\left(|u_{t-1}|-\sqrt{2/\pi}\right)+\gamma u_{t-1}+\beta\log\sigma^2_{t-1} \\
	\epsilon_t&\sim \mathcal{N}(0,1)
\end{align*}

In [ ]:
forecaster = ARCH(mean='AR', lags=2, vol='EGARCH', p=1, o=1, q=1)  
forecaster.fit(y)

forecaster.summary()

## Forecasting

build a 5-period forecast using the AR(2)-GARCH(2,2,2) model. Assume that normalized shocks have a _t_ distribution

Specification

\begin{align*}
	y_t&=\alpha_0+\phi_1 y_{t-1}+\phi_2 y_{t-2}+u_t \\ u_t&=\sigma_t \epsilon_t \\
	\sigma^\lambda_t &=\omega+\alpha_1 |u_{t-1}|^\lambda+\alpha_2 |u_{t-2}|^\lambda+\gamma_1|u_{t-1}|^\lambda I(u_{t-1}<0)+\gamma_2|u_{t-2}|^\lambda I(u_{t-2}<0)+
	\beta_1\sigma^\lambda_{t-1}+\beta_2\sigma^\lambda_{t-2} \\
	\epsilon_t&\sim t
\end{align*}

where $\lambda=2$

In [ ]:
forecaster = ARCH(mean='AR', lags=2, vol='GARCH', p=2, o=2, q=2, power=2, dist='t')

# set the forecasting horizon and frequency
fh = ForecastingHorizon(np.arange(1,6), freq ='W-Fri')

y_pred = forecaster.fit_predict(y=y, fh=fh)
y_pred

In [ ]:
# volatility forecast
forecaster.predict_var(fh=fh)